# 110 — SAR ships: SSDD and HRSID

Radar is a different sensor altogether. This lesson takes the optical **RTMDet** DOTA detector to synthetic aperture radar (SAR): it explores the two standard SAR ship benchmarks, measures how badly the optical model does zero-shot, then fine-tunes with the published **Faster R-CNN** 12-epoch recipes (no SAR Hub zoo for RTMDet/YOLO-OBB yet).

| | SSDD | HRSID |
|---|---|---|
| Chips | ~1,160, mixed sizes | 5,604, 800×800 |
| Ships | 2,456 | ~17,000 |
| Format | `dataset.format: ssdd` (VOC XML, COCO, or DOTA layouts) | `dataset.format: hrsid` (COCO JSON) |
| Split | official: file numbers ending in 1 or 9 are test | official train / test JSONs |
| Recipe | `rotated_faster_rcnn/ssdd_le90_1x.json`, keep-ratio 608 | `rotated_faster_rcnn/hrsid_le90_1x.json`, keep-ratio 800 |
| Published (held-out test mAP50) | **90.34%** | **78.55%** |
| Get it | Auto-download below from [Official-SSDD](https://github.com/TianwenZhang0825/Official-SSDD) — authors ship a **RAR** (`Official-SSDD-OPEN.rar`; or set `SSDD_DATA_ROOT` / attach a Kaggle dump whose name contains `ssdd`) | [HRSID](https://github.com/chaozhong2010/HRSID) → `HRSID_DATA_ROOT` (`HRSID_JPG`) or attach a Kaggle dump |

Either dataset is enough; the lesson uses whichever it finds. **SSDD** is fetched automatically when missing (Internet ON; needs `unrar`/`unar`/`7z` to unpack). HRSID still needs a local dump or Kaggle input. [Live notebook](https://www.kaggle.com/code/jeffaudi/orienteddet-110-sar-ships-ssdd-and-hrsid). Post: [v0.3.0: four datasets and ONNX](https://deeplearning.earth/posts/2026-09-21_oriented-det_v0_3_0_four_datasets_and_onnx/)


## What changes with radar

A SAR satellite sends microwave pulses and records what bounces back, so it sees through clouds and at night. The image is **backscatter intensity**, a single channel (replicated to three for an RGB backbone), with a grainy multiplicative noise called **speckle**. Ships are bright, because metal superstructures reflect strongly back to the sensor, and open sea is dark. Offshore scenes are therefore easy. **Inshore** scenes are hard: harbours, piers, and buildings are just as bright as ships.

The optical DOTA model has never seen any of this, yet it is a good starting point: its low-level filters and its idea of what an elongated ship looks like transfer. The zero-shot baseline below uses Hub `rotated_rtmdet_dota_le90_3x`. Both published recipes fine-tune from `hf://rotated_faster_rcnn_dota_le90_1x` with `keep_ratio` resizing (lesson 108) — SAR chips are small already, so there is **no tiling**.


## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.4.3"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"


In [ ]:
import os, subprocess, sys, time, urllib.request, warnings
from pathlib import Path

# Quiet Kaggle / TensorFlow absl noise and tqdm's missing-ipywidgets hint.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input dataset dir whose name contains one of `needles` (None off Kaggle).

    Searches a few levels deep so mounts like `/kaggle/input/datasets/<owner>/<slug>` match.
    """
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    dirs = []

    def walk(p: Path, depth: int):
        if depth > 3:
            return
        try:
            kids = list(p.iterdir())
        except OSError:
            return
        for k in kids:
            if k.is_dir():
                dirs.append(k)
                walk(k, depth + 1)

    walk(root, 0)
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 101 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")



## 1. Point at the data

In [ ]:
from oriented_det.data.hrsid import HRSIDDataset
from oriented_det.data.ssdd import SSDDDataset, resolve_ssdd_root
import shutil
import zipfile

RUN_SMOKE = False  # True: a short fine-tuning run on a GPU (a few minutes)
RUN_FULL = False   # True: run the full published 1x recipe (GPU, long)

# Authors' Google Drive file from https://github.com/TianwenZhang0825/Official-SSDD
# (Official-SSDD-OPEN.rar — not a zip)
SSDD_GDRIVE_ID = "1glNJUGotrbEyk43twwB9556AdngJsynZ"
SSDD_MIN_ARCHIVE_BYTES = 1_000_000


def _kaggle_dirs_matching(*needles):
    root = Path("/kaggle/input")
    if not root.is_dir():
        return []
    found = []

    def walk(p: Path, depth: int):
        if depth > 4:
            return
        try:
            kids = list(p.iterdir())
        except OSError:
            return
        for k in kids:
            if not k.is_dir():
                continue
            name = k.name.lower()
            if any(n.lower() in name for n in needles):
                found.append(k)
            walk(k, depth + 1)

    walk(root, 0)
    return found


def _archive_kind(path: Path) -> str | None:
    """Return 'zip', 'rar', or None from magic bytes (Drive often serves HTML on failure)."""
    try:
        with path.open("rb") as f:
            head = f.read(8)
    except OSError:
        return None
    if head.startswith(b"PK"):
        return "zip"
    if head.startswith(b"Rar!\x1a\x07") or head.startswith(b"Rar!"):
        return "rar"
    return None


def _ensure_unrar():
    """Return (exe_name, path), installing unrar on Debian/Kaggle if needed."""
    for exe in ("unrar", "unar", "7z"):
        path = shutil.which(exe)
        if path:
            return exe, path
    print("installing unrar (needed for Official-SSDD-OPEN.rar)…", flush=True)
    last = None
    for prefix in ([], ["sudo"]):
        subprocess.run([*prefix, "apt-get", "update", "-qq"], capture_output=True, text=True)
        last = subprocess.run(
            [*prefix, "apt-get", "install", "-y", "-qq", "unrar"],
            capture_output=True, text=True,
        )
        path = shutil.which("unrar")
        if path:
            return "unrar", path
    raise RuntimeError(
        "unrar is not available and apt install failed. "
        "Install unrar/unar/7z, or download Official-SSDD manually and set SSDD_DATA_ROOT.\n"
        + ((last.stderr or last.stdout or "") if last else "")[:500]
    )


def _extract_ssdd_archive(archive: Path, extract_dir: Path) -> None:
    kind = _archive_kind(archive)
    print(f"extracting {archive.name} ({kind or 'unknown'}) -> {extract_dir}…", flush=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    if kind == "zip" or (kind is None and zipfile.is_zipfile(archive)):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(extract_dir)
        return
    if kind == "rar" or archive.suffix.lower() == ".rar":
        exe, _ = _ensure_unrar()
        if exe == "unrar":
            cmd = ["unrar", "x", "-o+", str(archive), str(extract_dir) + "/"]
        elif exe == "unar":
            cmd = ["unar", "-f", "-o", str(extract_dir), str(archive)]
        else:
            cmd = ["7z", "x", f"-o{extract_dir}", "-y", str(archive)]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0:
            raise FileNotFoundError(
                f"Could not unpack {archive}: {(proc.stderr or proc.stdout or '')[:500]}"
            )
        return
    raise FileNotFoundError(
        f"Could not unpack {archive} (not a zip/rar; often a failed Google Drive HTML page). "
        "Delete it, re-run with Internet ON, or download Official-SSDD manually and set SSDD_DATA_ROOT."
    )


def ensure_ssdd_root():
    """Local / Kaggle SSDD, else download Official-SSDD.rar from the authors' Google Drive."""
    candidates = []
    if os.environ.get("SSDD_DATA_ROOT"):
        candidates.append(Path(os.environ["SSDD_DATA_ROOT"]))
    for needle in ("official-ssdd", "rbox-ssdd", "rbox_ssdd", "ssdd"):
        hit = kaggle_input(needle)
        if hit is not None:
            candidates.append(hit)
    candidates.extend(_kaggle_dirs_matching("official-ssdd", "rbox-ssdd", "ssdd"))
    candidates.append(WORK / "Official-SSDD-OPEN")
    candidates.append(WORK / "ssdd_download")

    seen = set()
    for raw in candidates:
        raw = Path(raw)
        key = str(raw.resolve()) if raw.exists() else str(raw)
        if key in seen or not raw.exists():
            continue
        seen.add(key)
        try:
            root = resolve_ssdd_root(raw)
            print(f"using SSDD at {root}")
            return root
        except FileNotFoundError:
            continue

    print(
        "SSDD not found locally; downloading Official-SSDD from Google Drive "
        "(Internet ON; one-time, cached under the work dir)…",
        flush=True,
    )
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    # Authors ship a RAR; older notebook versions cached a misnamed .zip — drop junk.
    archive = WORK / "Official-SSDD-OPEN.rar"
    stale_zip = WORK / "Official-SSDD-OPEN.zip"
    if stale_zip.is_file() and _archive_kind(stale_zip) not in {"zip", "rar"}:
        print(f"removing invalid cached {stale_zip.name}", flush=True)
        stale_zip.unlink()
    elif stale_zip.is_file() and not archive.is_file():
        archive = stale_zip  # reuse if it actually is zip/rar under the old name

    extract_dir = WORK / "ssdd_download"
    need_download = (
        not archive.is_file()
        or archive.stat().st_size < SSDD_MIN_ARCHIVE_BYTES
        or _archive_kind(archive) is None
    )
    if need_download:
        if archive.is_file():
            archive.unlink()
        url = f"https://drive.google.com/uc?id={SSDD_GDRIVE_ID}"
        # Prefer the real Drive filename (*.rar); fall back to our path.
        out = gdown.download(url, str(archive), quiet=False, fuzzy=True)
        if out and Path(out).is_file() and Path(out).resolve() != archive.resolve():
            Path(out).replace(archive)

    kind = _archive_kind(archive) if archive.is_file() else None
    if not archive.is_file() or archive.stat().st_size < SSDD_MIN_ARCHIVE_BYTES or kind is None:
        raise FileNotFoundError(
            "Official-SSDD download failed (empty/missing/HTML stub). "
            "Set SSDD_DATA_ROOT, attach a Kaggle dump whose name contains 'ssdd', "
            "or download manually from https://github.com/TianwenZhang0825/Official-SSDD"
        )

    extract_dir.mkdir(parents=True, exist_ok=True)
    try:
        root = resolve_ssdd_root(extract_dir)
        print(f"using cached extract {root}")
        return root
    except FileNotFoundError:
        pass

    try:
        _extract_ssdd_archive(archive, extract_dir)
    except Exception as exc:
        raise FileNotFoundError(
            f"Could not unpack {archive} ({exc}). "
            "Download Official-SSDD manually and set SSDD_DATA_ROOT."
        ) from exc

    root = resolve_ssdd_root(extract_dir)
    print(f"using SSDD at {root}")
    return root


def ensure_hrsid_root():
    """Local / Kaggle HRSID only (no auto-download — large COCO dump)."""
    candidates = []
    if os.environ.get("HRSID_DATA_ROOT"):
        candidates.append(Path(os.environ["HRSID_DATA_ROOT"]))
    for needle in ("hrsid-jpg", "hrsid"):
        hit = kaggle_input(needle)
        if hit is not None:
            candidates.append(hit)
    candidates.extend(_kaggle_dirs_matching("hrsid"))
    for raw in candidates:
        raw = Path(raw)
        if raw.is_dir():
            return raw
    return Path(os.environ.get("HRSID_DATA_ROOT", "/path/to/data/HRSID_JPG"))


SETS = {
    "ssdd": dict(
        loader=SSDDDataset,
        root=ensure_ssdd_root(),
        recipe="rotated_faster_rcnn/ssdd_le90_1x.json",
        published=90.34,
    ),
    "hrsid": dict(
        loader=HRSIDDataset,
        root=ensure_hrsid_root(),
        recipe="rotated_faster_rcnn/hrsid_le90_1x.json",
        published=78.55,
    ),
}
data = {}
for name, s in SETS.items():
    if Path(s["root"]).is_dir():
        try:
            data[name] = {
                split: s["loader"](s["root"], split=split, difficult_strategy="keep")
                for split in ("train", "test")
            }
            print(f"{name}: train {len(data[name]['train'])}, test {len(data[name]['test'])} chips  ({s['root']})")
        except Exception as exc:
            print(f"{name}: found at {s['root']} but failed to load ({exc})")
    else:
        print(f"{name}: not found at {s['root']}")
if not data:
    raise FileNotFoundError(
        "Neither SSDD nor HRSID loaded. SSDD should auto-download with Internet ON; "
        "for HRSID set HRSID_DATA_ROOT or attach a Kaggle dump."
    )
SAR = "ssdd" if "ssdd" in data else next(iter(data))
print("using", SAR, "for the zero-shot test and fine-tuning")


## 2. Ships are small

Box length in pixels, per dataset. Compare with HRSC2016 (lesson 108), where the median ship spans a few hundred pixels.


In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, splits in data.items():
    lengths = [max(a.rbox.width, a.rbox.height) for s in splits["train"] for a in s.annotations]
    ax.hist(lengths, bins=np.linspace(0, 200, 41), alpha=0.6, label=f"{name} (median {np.median(lengths):.0f} px)")
    print(f"{name}: {len(lengths)} training ships, {len(lengths) / len(splits['train']):.1f} per chip")
ax.set_xlabel("ship length (px)"); ax.legend(); plt.show()

## 3. One chip each

In [ ]:
from PIL import Image
from oriented_det.utils.viz import DrawingSpec, draw_boxes

views, titles = [], []
for name, splits in data.items():
    s = max(list(splits["train"])[:50], key=lambda s: len(s.annotations))  # a busy chip
    img = Image.open(s.image_path).convert("RGB")
    views.append(draw_boxes(img, [a.rbox for a in s.annotations],
                            specs=[DrawingSpec(outline=(0, 255, 128), width=2)] * len(s.annotations)))
    titles.append(f"{name}: {s.image_path.name}, {len(s.annotations)} ships")
show(*views, titles=titles, size=6)

## 4. Zero-shot: the optical model on radar

Run the DOTA **RTMDet** Hub checkpoint on test chips, keep only its `ship` class, and compute mAP50. This is the baseline fine-tuning has to beat.


In [ ]:
from oriented_det.data import Detection, GroundTruth, compute_oriented_map


def evaluate(model, config, class_names, samples, keep_class="ship"):
    """mAP50 of one class over dataset samples, keeping every detection down to score 0.05."""
    dets, gts = {}, {}
    for s in samples:
        gts[s.image_path.stem] = [GroundTruth(a.rbox, 0, keep_class, a.difficult, s.image_path.stem) for a in s.annotations]
        found = detect(model, config, class_names, Image.open(s.image_path).convert("RGB"), score_thr=0.05, classes=[keep_class])
        dets[s.image_path.stem] = [Detection(d["rbox"], float(d["score"]), 0, keep_class, s.image_path.stem) for d in found]
    return compute_oriented_map(dets, gts, class_names=[keep_class], show_progress=False)[0]

In [ ]:
N_TEST = 100  # test chips used for both evaluations below; None = all
subset = list(data[SAR]["test"])[:N_TEST]
dota_model, dota_config, dota_classes = hub_model("rotated_rtmdet_dota_le90_3x")
zero_shot = evaluate(dota_model, dota_config, dota_classes, subset)
print(f"zero-shot RTMDet DOTA on {len(subset)} {SAR} test chips: mAP50 {100 * zero_shot:.2f}% "
      f"(published after fine-tuning: {SETS[SAR]['published']}%)")


## 5. Fine-tune

The published recipe, patched for a smoke run: 32 chips, one epoch. The full recipe trains all chips for 12 epochs (learning-rate drops at epochs 8 and 11).

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

cfg = TrainingExperimentConfig.load(CONFIGS / SETS[SAR]["recipe"])
print("recipe:", SETS[SAR]["recipe"], "| init from", cfg.checkpoint.load_from_checkpoint,
      "|", cfg.preprocessing.resize_mode, cfg.preprocessing.target_size)
cfg.dataset.data_root = SETS[SAR]["root"]
cfg.dataset.max_train_samples = 32
cfg.dataset.max_val_samples = 16
cfg.training.num_epochs = 1
cfg.evaluation.compute_map_every_n_epochs = 1
smoke = WORK / f"110_{SAR}_smoke.json"
cfg.save(smoke)
print("wrote", smoke)


In [ ]:
def newest_run(model_type):
    """(checkpoint, config) of the most recent odet train run for this model type."""
    run_dir = max((WORK / "runs" / model_type).iterdir(), key=lambda p: p.stat().st_mtime)
    ckpt = max((run_dir / "checkpoints").glob("*.pth"), key=lambda p: p.stat().st_mtime)
    return ckpt, run_dir / "config.json"

In [ ]:
import gc

import torch
from oriented_det.runtime.checkpoint import load_model_from_checkpoint

if not RUN_SMOKE:
    print("RUN_SMOKE=False: skipping training. Set it to True in section 1 on a GPU.")
else:
    dota_model = None  # hand the GPU memory to the training subprocess
    gc.collect(); torch.cuda.empty_cache()
    run([*ODET, "train", "--config", smoke])
    ckpt, run_config = newest_run("rotated_faster_rcnn")
    tuned = load_model_from_checkpoint(str(ckpt), str(run_config), device=DEVICE)
    print(f"after the smoke fine-tune: mAP50 {100 * evaluate(*tuned, subset):.2f}% vs zero-shot {100 * zero_shot:.2f}%")

## 6. The full recipes

```bash
odet train --config oriented_det/configs/rotated_faster_rcnn/ssdd_le90_1x.json    # set dataset.data_root first
odet train --config oriented_det/configs/rotated_faster_rcnn/hrsid_le90_1x.json
make eval-val EXPERIMENT=runs/rotated_faster_rcnn/<timestamp>                       # held-out official test
```

Or set `RUN_FULL = True` above and run the next cell to launch the matching recipe from this notebook.

Published held-out test mAP50 after the full 12 epochs: **90.34%** on SSDD (literature Faster R-CNN is often ~89%) and **78.55%** on HRSID. HRSID is harder: more inshore scenes, more small ships. Details: [SSDD](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#ssdd) and [HRSID](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#hrsid) in the data guide. There are no SAR checkpoints on the Hub yet.

## Optional: export SSDD to DOTA layout

Uncomment to write DOTA-format folders (useful if you prefer `odet tile-dota` tooling). Not required for the native `ssdd` recipe.


In [ ]:
if not RUN_FULL:
    print("RUN_FULL=False: skipping full 1x train (use CLI or set RUN_FULL=True)")
else:
    recipe = CONFIGS / SETS[SAR]["recipe"]
    cfg = TrainingExperimentConfig.load(recipe)
    cfg.dataset.data_root = SETS[SAR]["root"]
    full_path = WORK / f"110_{SAR}_full.json"
    cfg.save(full_path)
    run([*ODET, "train", "--config", full_path])


In [ ]:
# Optional SSDD → DOTA export (uncomment to run).
# from oriented_det.data.ssdd import export_ssdd_to_dota
# if SAR == "ssdd":
#     out = WORK / "SSDD-dota"
#     export_ssdd_to_dota(SETS["ssdd"]["root"], out)
#     print("wrote", out)
# else:
#     print("export_ssdd_to_dota only applies to SSDD")
print("SSDD→DOTA export cell ready (uncomment to run)")


## What to try next

- Split the SSDD test set into inshore and offshore chips (the official release lists them) and evaluate each: most of the error lives inshore.
- Train the Oriented R-CNN HRSID recipe (`oriented_rcnn/hrsid_le90_1x.json`) and compare with Faster R-CNN.
- Compare zero-shot `rotated_faster_rcnn_dota_le90_1x` with the RTMDet baseline on the same chips.
- **Next:** [111 — Export RTMDet to ONNX](111_onnx_export.ipynb).
